# StockTwits Data Cleaning Pipeline

This notebook processes raw StockTwits message data and applies the following cleaning steps:

1. **Filter for non-missing sentiment** - Keep only messages with sentiment values (Bullish/Bearish)
2. **Filter for single symbols** - Keep only messages mentioning exactly one stock symbol
3. **Extract symbol** - Convert symbol_list to a clean symbol column
4. **Save cleaned data** - Export to a new folder

**Input:** `dataset/v1/data/csv/feature_wo_messages/`  
**Output:** `dataset/v1/data/csv/feature_wo_messages_cleaned/`

## 1. Setup and Configuration

In [ ]:
"""
StockTwits Dataset Downloader

This script systematically downloads the StockTwits dataset from the AWS S3 bucket
(s3://stocktwits-nyu) to the local directory, maintaining the folder structure.

The dataset includes:
- feature_wo_messages: Features extracted without message content
- messages: Raw message data
- msg_info: Metadata for messages
- sentiments: Sentiment analysis results
- symbols: Information about stock symbols
- symbol_sentiments: Sentiment information mapped to symbols
"""

import subprocess
import sys
import os
from pathlib import Path


def check_aws_cli():
    """Check if AWS CLI is installed."""
    # Get the Python executable path
    python_exe = sys.executable
    
    try:
        result = subprocess.run(
            [python_exe, "-m", "awscli", "--version"],
            capture_output=True,
            text=True,
            check=True
        )
        print(f"✓ AWS CLI found: {result.stdout.strip()}")
        return True
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("✗ AWS CLI not found.")
        print("\nPlease install AWS CLI:")
        print("  - Download from: https://aws.amazon.com/cli/")
        print("  - Or use: pip install awscli")
        return False


def list_s3_contents(s3_path):
    """List contents of an S3 path."""
    python_exe = sys.executable
    try:
        print(f"\nListing contents of: {s3_path}")
        result = subprocess.run(
            [python_exe, "-m", "awscli", "s3", "ls", "--no-sign-request", s3_path],
            capture_output=True,
            text=True,
            check=True
        )
        print(result.stdout)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error listing S3 contents: {e.stderr}")
        return False


def download_dataset(base_path=".", create_subdirs=True):
    """
    Download the entire StockTwits dataset from S3.
    
    Args:
        base_path: Local directory where data will be downloaded
        create_subdirs: If True, creates dataset/v1/data/csv structure
    """
    # S3 bucket configuration
    BASE_URL = "s3://stocktwits-nyu"
    CSV_URL = f"{BASE_URL}/dataset/v1/data/csv"
    
    # Prepare local directory
    if create_subdirs:
        local_path = Path(base_path) / "dataset" / "v1" / "data" / "csv"
    else:
        local_path = Path(base_path)
    
    local_path.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"StockTwits Dataset Downloader")
    print(f"{'='*60}")
    print(f"Source: {CSV_URL}")
    print(f"Destination: {local_path.absolute()}")
    print(f"{'='*60}\n")
    
    # Check AWS CLI availability
    if not check_aws_cli():
        return False
    
    # List available data folders
    print("\n" + "="*60)
    print("Available data categories:")
    print("="*60)
    list_s3_contents(f"{CSV_URL}/")
    
    # Data categories to download
    categories = [
        "feature_wo_messages",
        "messages",
        "msg_info",
        "sentiments",
        "symbols",
        "symbol_sentiments"
    ]
    
    # Download each category
    print("\n" + "="*60)
    print("Starting download...")
    print("="*60 + "\n")
    
    for category in categories:
        s3_category_path = f"{CSV_URL}/{category}/"
        local_category_path = local_path / category
        
        print(f"\n{'─'*60}")
        print(f"Downloading: {category}")
        print(f"{'─'*60}")
        
        try:
            # Use aws s3 sync to download all files in the category
            python_exe = sys.executable
            result = subprocess.run(
                [
                    python_exe, "-m", "awscli",
                    "s3", "sync",
                    "--no-sign-request",
                    s3_category_path,
                    str(local_category_path)
                ],
                capture_output=True,
                text=True,
                check=True
            )
            
            # Print download progress
            if result.stdout:
                print(result.stdout)
            if result.stderr:
                print(result.stderr)
            
            # Count downloaded files
            if local_category_path.exists():
                file_count = len(list(local_category_path.glob("*.csv")))
                print(f"✓ {category}: {file_count} files downloaded")
            
        except subprocess.CalledProcessError as e:
            print(f"✗ Error downloading {category}: {e.stderr}")
            continue
    
    print("\n" + "="*60)
    print("Download Summary")
    print("="*60)
    
    # Print summary of downloaded files
    total_files = 0
    total_size = 0
    
    for category in categories:
        local_category_path = local_path / category
        if local_category_path.exists():
            files = list(local_category_path.glob("*.csv"))
            category_size = sum(f.stat().st_size for f in files)
            total_files += len(files)
            total_size += category_size
            
            size_mb = category_size / (1024 * 1024)
            print(f"{category:25} {len(files):4} files  {size_mb:8.2f} MB")
    
    print("─"*60)
    total_size_gb = total_size / (1024 * 1024 * 1024)
    print(f"{'TOTAL':25} {total_files:4} files  {total_size_gb:8.2f} GB")
    print("="*60)
    print(f"\n✓ Download complete! Data saved to: {local_path.absolute()}")
    
    return True


def download_single_category(category, base_path="."):
    """
    Download a single category of data.
    
    Args:
        category: One of the data categories (e.g., 'messages', 'sentiments')
        base_path: Local directory where data will be downloaded
    """
    BASE_URL = "s3://stocktwits-nyu"
    CSV_URL = f"{BASE_URL}/dataset/v1/data/csv"
    
    local_path = Path(base_path) / "dataset" / "v1" / "data" / "csv" / category
    local_path.mkdir(parents=True, exist_ok=True)
    
    print(f"\nDownloading {category} to {local_path.absolute()}...")
    
    if not check_aws_cli():
        return False
    
    s3_category_path = f"{CSV_URL}/{category}/"
    
    python_exe = sys.executable
    try:
        result = subprocess.run(
            [
                python_exe, "-m", "awscli",
                "s3", "sync",
                "--no-sign-request",
                s3_category_path,
                str(local_path)
            ],
            capture_output=True,
            text=True,
            check=True
        )
        
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        
        file_count = len(list(local_path.glob("*.csv")))
        print(f"✓ Downloaded {file_count} files")
        return True
        
    except subprocess.CalledProcessError as e:
        print(f"✗ Error: {e.stderr}")
        return False


if __name__ == "__main__":
    # You can modify these parameters:
    # - base_path: Where to save the data (default: current directory)
    # - create_subdirs: Whether to create dataset/v1/data/csv structure (default: True)
    
    # Download all data
    success = download_dataset(base_path=".", create_subdirs=True)
    
    # Alternative: Download only specific categories
    # Uncomment the following to download only specific categories:
    # download_single_category("messages", base_path=".")
    # download_single_category("sentiments", base_path=".")
    
    if success:
        print("\n✓ All downloads completed successfully!")
    else:
        print("\n✗ Download encountered errors. Please check the output above.")
        sys.exit(1)

In [2]:
import os
import ast
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Define paths
input_folder = r"c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages"
output_folder = r"c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_cleaned"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Get all CSV files in the input folder
csv_files = sorted([f for f in os.listdir(input_folder) if f.endswith('.csv')])

print(f"Found {len(csv_files)} CSV files to process")
print(f"Output folder: {output_folder}")

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Found 248 CSV files to process
Output folder: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_cleaned


## 2. Define Cleaning Function

In [ ]:
def clean_dataframe(df):
    """
    Clean a dataframe by:
    1. Filtering for non-missing sentiment
    2. Filtering for exactly one symbol
    3. Extracting the symbol as a string column
    4. Dropping the symbol_list columns
    """
    # Filter for non-missing sentiment
    df_cleaned = df[df['sentiment'].notna()].copy()
    
    # Parse symbol_list and filter for exactly one symbol
    df_cleaned['symbol_list_parsed'] = df_cleaned['symbol_list'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df_cleaned = df_cleaned[df_cleaned['symbol_list_parsed'].apply(
        lambda x: isinstance(x, list) and len(x) == 1
    )]
    
    # Extract the single symbol
    df_cleaned['symbol'] = df_cleaned['symbol_list_parsed'].apply(lambda x: x[0])
    
    # Drop the symbol_list columns
    df_cleaned = df_cleaned.drop(columns=['symbol_list', 'symbol_list_parsed'])
    
    return df_cleaned

Test: Original rows: 2058472, Cleaned rows: 359510
Columns: ['message_id', 'user_id', 'created_at', 'sentiment', 'parent_message_id', 'in_reply_to_message_id', 'symbol']


## 3. Process All Files

In [ ]:
# Process all files
total_original_rows = 0
total_cleaned_rows = 0
processed_files = 0

print("Processing files...")
for csv_file in tqdm(csv_files):
    try:
        # Read the file
        input_path = os.path.join(input_folder, csv_file)
        df = pd.read_csv(input_path)
        
        # Clean the dataframe
        df_cleaned = clean_dataframe(df)
        
        # Save to output folder
        output_path = os.path.join(output_folder, csv_file)
        df_cleaned.to_csv(output_path, index=False)
        
        # Update statistics
        total_original_rows += len(df)
        total_cleaned_rows += len(df_cleaned)
        processed_files += 1
        
    except Exception as e:
        print(f"\nError processing {csv_file}: {e}")

print(f"\n{'='*60}")
print(f"Processing complete!")
print(f"Files processed: {processed_files}/{len(csv_files)}")
print(f"Total original rows: {total_original_rows:,}")
print(f"Total cleaned rows: {total_cleaned_rows:,}")
print(f"Reduction: {(1 - total_cleaned_rows/total_original_rows)*100:.2f}%")
print(f"Output folder: {output_folder}")

Processing files...


100%|██████████| 248/248 [21:53<00:00,  5.29s/it]


Processing complete!
Files processed: 248/248
Total original rows: 501,442,290
Total cleaned rows: 106,913,064
Reduction: 78.68%
Output folder: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_cleaned


## 4. Estimate Memory Requirements

In [14]:
import sys

# Get list of cleaned CSV files
cleaned_files = sorted([f for f in os.listdir(output_folder) if f.endswith('.csv')])

# Sample a few files to estimate memory usage
sample_size = 5
sample_files = cleaned_files[:sample_size]

print(f"Sampling {sample_size} files to estimate memory usage...\n")

total_sample_rows = 0
total_sample_memory = 0

for csv_file in sample_files:
    file_path = os.path.join(output_folder, csv_file)
    df_sample = pd.read_csv(file_path)
    
    memory_usage = df_sample.memory_usage(deep=True).sum()
    total_sample_rows += len(df_sample)
    total_sample_memory += memory_usage
    
    print(f"{csv_file}: {len(df_sample):,} rows, {memory_usage / (1024**2):.2f} MB")

# Calculate average memory per row
avg_memory_per_row = total_sample_memory / total_sample_rows

# Estimate total memory needed
total_rows = total_cleaned_rows  # From previous processing step
estimated_memory = total_rows * avg_memory_per_row

print(f"\n{'='*60}")
print(f"Memory Estimation:")
print(f"{'='*60}")
print(f"Average memory per row: {avg_memory_per_row:.2f} bytes")
print(f"Total rows (cleaned): {total_rows:,}")
print(f"Estimated memory needed: {estimated_memory / (1024**2):.2f} MB ({estimated_memory / (1024**3):.2f} GB)")
print(f"\nWith 50% overhead for operations: {estimated_memory * 1.5 / (1024**3):.2f} GB")
print(f"With 100% overhead for operations: {estimated_memory * 2 / (1024**3):.2f} GB")

Sampling 5 files to estimate memory usage...

feature_wo_messages_000.csv: 359,510 rows, 80.17 MB
feature_wo_messages_001.csv: 362,861 rows, 80.90 MB
feature_wo_messages_002.csv: 363,558 rows, 81.12 MB
feature_wo_messages_003.csv: 364,857 rows, 81.46 MB
feature_wo_messages_004.csv: 366,075 rows, 81.73 MB

Memory Estimation:
Average memory per row: 233.95 bytes
Total rows (cleaned): 106,913,064
Estimated memory needed: 23854.02 MB (23.29 GB)

With 50% overhead for operations: 34.94 GB
With 100% overhead for operations: 46.59 GB


## 5. Load and Merge All Cleaned Files

In [3]:
import gc

# Get all cleaned CSV files
cleaned_files = sorted([f for f in os.listdir(output_folder) if f.endswith('.csv')])

print(f"Loading {len(cleaned_files)} files...")
print(f"This may take several minutes...\n")

# Load all files and concatenate
dfs = []
for i, csv_file in enumerate(tqdm(cleaned_files)):
    file_path = os.path.join(output_folder, csv_file)
    df_temp = pd.read_csv(file_path)
    df_temp = df_temp.drop(columns=['parent_message_id', 'in_reply_to_message_id'])
    dfs.append(df_temp)
    
    # Progress update every 50 files
    if (i + 1) % 50 == 0:
        current_memory = sum(df.memory_usage(deep=True).sum() for df in dfs)
        print(f"  Loaded {i+1}/{len(cleaned_files)} files - Current memory: {current_memory / (1024**3):.2f} GB")

print("\nConcatenating all dataframes...")
df_merged = pd.concat(dfs, ignore_index=True)

# Clear the list to free memory
del dfs
gc.collect()

print(f"\n{'='*60}")
print(f"Merge Complete!")
print(f"{'='*60}")
print(f"Total rows: {len(df_merged):,}")
print(f"Total columns: {len(df_merged.columns)}")
print(f"Memory usage: {df_merged.memory_usage(deep=True).sum() / (1024**3):.2f} GB")
print(f"\nDataframe info:")
print(df_merged.info())

Loading 248 files...
This may take several minutes...



 20%|██        | 50/248 [00:15<05:04,  1.54s/it]

  Loaded 50/248 files - Current memory: 3.73 GB


 40%|████      | 100/248 [00:36<07:46,  3.15s/it]

  Loaded 100/248 files - Current memory: 8.20 GB


...

  Loaded 150/248 files - Current memory: 13.33 GB


 81%|████████  | 200/248 [01:38<05:18,  6.63s/it]

  Loaded 200/248 files - Current memory: 17.97 GB


100%|██████████| 248/248 [01:48<00:00,  2.29it/s]




Concatenating all dataframes...

Merge Complete!
Total rows: 106,913,064
Total columns: 5
Memory usage: 21.70 GB

Dataframe info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106913064 entries, 0 to 106913063
Data columns (total 5 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   message_id  int64 
 1   user_id     int64 
 2   created_at  object
 3   sentiment   object
 4   symbol      object
dtypes: int64(2), object(3)
memory usage: 4.0+ GB
None

Merge Complete!
Total rows: 106,913,064
Total columns: 5
Memory usage: 21.70 GB

Dataframe info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106913064 entries, 0 to 106913063
Data columns (total 5 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   message_id  int64 
 1   user_id     int64 
 2   created_at  object
 3   sentiment   object
 4   symbol      object
dtypes: int64(2), object(3)
memory usage: 4.0+ GB
None


# 6. Find the first market close after each tweet

In [4]:
# I don't know if created_at from df is timezone aware or not. I wrote my code
# to work with both contingencies. I convert the creation time to US/Eastern
# and get rid of the timezone to make things easier.
import datetime as dt
import pandas_datareader as port

print("\n Converting created_at to US/Eastern timezone...")
try:
    df_merged["created_at"] = pd.to_datetime(df_merged["created_at"]).dt.tz_convert("US/Eastern").dt.tz_localize(None) 
except:
    df_merged["created_at"] = pd.to_datetime(df_merged["created_at"]).dt.tz_localize("UTC").dt.tz_convert("US/Eastern").dt.tz_localize(None) 

# extract date and time components
df_merged['date'] = pd.to_datetime(df_merged['created_at'].dt.date) # a nuisance required for the merge command to work
df_merged["time"] = df_merged["created_at"].dt.time
df_merged["after_hour"] = df_merged["time"] > dt.time(16,0,0)

# find the first market close after a tweet
print("\n Working with the Fama-French data...")
ff = port.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench',
                     start=dt.date(2008, 1, 1),
                     end=dt.date(2024, 12, 31))[0].reset_index().rename(columns={"Date":"first_close"})
ff["second_close"] = ff["first_close"].shift(-1)
trading_days = pd.date_range(start="2008-1-1", end="2024-12-31").to_frame(name="date")
trading_days = pd.merge(trading_days, ff, left_on="date", right_on="first_close", how="left", indicator=True)
trading_days["business_day"] = trading_days["_merge"] == "both"
trading_days = trading_days[["date", "business_day", "first_close", "second_close"]]
trading_days = trading_days.bfill()

# merge with our main dataframe
print("\n Merging trading days with main dataframe...")
df_merged = pd.merge(df_merged, trading_days, on="date", how="left") # clean merge

# find the first close after the tweet

df_merged["first_close_after_tweet"] = df_merged["first_close"]
I = df_merged["business_day"] & df_merged["after_hour"]
df_merged.loc[I, "first_close_after_tweet"] = df_merged.loc[I, "second_close"]

# drop the extra columns
df_merged = df_merged.drop(columns=["date", "time", "after_hour", "business_day", "first_close", "second_close"])

# rename the first_close_after_tweet. This is our official date henceforth.
df_merged = df_merged.rename(columns={"first_close_after_tweet":"date"})


 Converting created_at to US/Eastern timezone...

 Working with the Fama-French data...

 Working with the Fama-French data...


C:\Users\skazempour\AppData\Local\Temp\ipykernel_20952\229755338.py:20: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff = port.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench',



 Merging trading days with main dataframe...


In [5]:
# Save merged data by year using the 'date' column; drop 'created_at' before saving
import os

# Output folder for yearly splits
output_by_year_folder = r"c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_by_year"
os.makedirs(output_by_year_folder, exist_ok=True)

# Ensure 'date' is datetime and drop 'created_at'
df_out = df_merged.copy()
df_out['date'] = pd.to_datetime(df_out['date'])
if 'created_at' in df_out.columns:
    df_out = df_out.drop(columns=['created_at'])

# Filter out rows with missing dates
df_out = df_out[df_out['date'].notna()]

# Derive year as integer
df_out['year'] = df_out['date'].dt.year.astype(int)
years = sorted(df_out['year'].unique())

print(f"Saving {len(years)} yearly files to: {output_by_year_folder}\n")
rows_written = 0
for y in years:
    df_y = df_out[df_out['year'] == y].drop(columns=['year'])
    out_path = os.path.join(output_by_year_folder, f"feature_wo_messages_{y}.csv")
    df_y.to_csv(out_path, index=False)
    rows_written += len(df_y)
    print(f"  {y}: {len(df_y):,} rows -> {out_path}")

print(f"\nDone. Total rows written: {rows_written:,}")

Saving 15 yearly files to: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_by_year

  2010: 16,228 rows -> c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_by_year\feature_wo_messages_2010.csv
  2011: 55,353 rows -> c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_by_year\feature_wo_messages_2011.csv
  2012: 103,409 rows -> c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_by_year\feature_wo_messages_2012.csv
  2012: 103,409 rows -> c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_by_year\feature_wo_messages_2012.csv
  2013: 586,177 rows -> c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_by_year\feature_wo_messages_2013.csv
  2013: 586,177 rows -> c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_by_year\feature_wo_messages_2013.csv
  2014: 1,410,083 rows -> c:\User

# 7. Merge with CRSP Data

Now we'll merge the yearly StockTwits files with CRSP data to add abnormal returns for multiple models and horizons.

In [3]:
import pandas as pd
import os
from tqdm import tqdm
import numpy as np

# Define paths
stocktwits_folder = r"c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_by_year"
crsp_folder = r"D:\CRSP"
output_folder = r"c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\merged_with_crsp"

# Create output folder
os.makedirs(output_folder, exist_ok=True)

# Define horizons we want
horizons = [1, 3, 5, 10, 21, 42, 63]

print(f"\n{'='*60}")
print("Starting year-by-year merge with abnormal return calculations...")
print(f"{'='*60}\n")

merge_stats = []

# Years to process (based on what we saved)
years = list(range(2010, 2025))

for year in tqdm(years, desc="Processing years"):
    stocktwits_file = os.path.join(stocktwits_folder, f"feature_wo_messages_{year}.csv")
    crsp_file = os.path.join(crsp_folder, f"dsf_final_{year}.pkl")
    
    # Check if files exist
    if not os.path.exists(stocktwits_file):
        print(f"  Skipping {year}: StockTwits file not found")
        continue
    if not os.path.exists(crsp_file):
        print(f"  Skipping {year}: CRSP file not found")
        continue
    
    # Load StockTwits data
    df_st = pd.read_csv(stocktwits_file)
    df_st['date'] = pd.to_datetime(df_st['date'])
    
    # Load CRSP data
    df_crsp = pd.read_pickle(crsp_file)
    df_crsp['date'] = pd.to_datetime(df_crsp['date'])
    
    # --- Calculate Abnormal Returns ---
    
    # 1. DGTW: Abnormal return = cumulative return - benchmark return
    for h in horizons:
        cumret_col = f'f_cumret{h}'
        dgtw_col = f'f_dgtw_ret{h}'
        if cumret_col in df_crsp.columns and dgtw_col in df_crsp.columns:
            df_crsp[f'ar_dgtw_{h}'] = df_crsp[cumret_col] - df_crsp[dgtw_col]
    
    # 2. CAPM, FF3, FF5, FF6: Cumulative abnormal returns
    # Sum individual abnormal returns from 1 to h
    models = ['capm', 'FF3', 'FF5', 'FF6']
    
    for model in models:
        # First, ensure we have all individual abnormal returns up to max horizon
        max_horizon = max(horizons)
        ar_cols = []
        for i in range(1, max_horizon + 1):
            col = f'f_{model}_ar{i}'
            if col in df_crsp.columns:
                ar_cols.append(col)
        
        # Calculate cumulative abnormal returns for each horizon
        if ar_cols:  # Only if we have the columns
            for h in horizons:
                # Sum from day 1 to day h
                cols_to_sum = [f'f_{model}_ar{i}' for i in range(1, h + 1) 
                              if f'f_{model}_ar{i}' in df_crsp.columns]
                if cols_to_sum:
                    df_crsp[f'ar_{model}_{h}'] = df_crsp[cols_to_sum].sum(axis=1)
    
    # Select columns for merge: permno, ticker, date, and all calculated abnormal returns
    cols_to_keep = ['permno', 'ticker', 'date']
    
    # Add all abnormal return columns
    for model in ['dgtw', 'capm', 'FF3', 'FF5', 'FF6']:
        for h in horizons:
            ar_col = f'ar_{model}_{h}'
            if ar_col in df_crsp.columns:
                cols_to_keep.append(ar_col)
    
    df_crsp_subset = df_crsp[cols_to_keep].copy()
    
    # Convert ticker to uppercase for matching (StockTwits uses uppercase symbols)
    df_crsp_subset['ticker'] = df_crsp_subset['ticker'].str.upper()
    
    # Merge on ticker and date
    df_merged = pd.merge(
        df_st,
        df_crsp_subset,
        left_on=['symbol', 'date'],
        right_on=['ticker', 'date'],
        how='left'
    )
    
    # Drop the redundant ticker column
    if 'ticker' in df_merged.columns:
        df_merged = df_merged.drop(columns=['ticker'])
    
    # Save merged data
    output_file = os.path.join(output_folder, f"merged_{year}.csv")
    df_merged.to_csv(output_file, index=False)
    
    # Track statistics
    match_rate = (df_merged['permno'].notna().sum() / len(df_merged)) * 100
    
    # Count how many abnormal return columns we successfully added
    ar_cols_in_merge = [col for col in df_merged.columns if col.startswith('ar_')]
    
    merge_stats.append({
        'year': year,
        'stocktwits_rows': len(df_st),
        'crsp_rows': len(df_crsp),
        'merged_rows': len(df_merged),
        'matched_rows': df_merged['permno'].notna().sum(),
        'match_rate': match_rate,
        'ar_columns': len(ar_cols_in_merge)
    })
    
    print(f"  {year}: {len(df_st):,} tweets -> {df_merged['permno'].notna().sum():,} matched ({match_rate:.1f}%), {len(ar_cols_in_merge)} AR columns")

# Summary statistics
print(f"\n{'='*60}")
print("Merge Summary")
print(f"{'='*60}\n")

df_stats = pd.DataFrame(merge_stats)
print(df_stats.to_string(index=False))

print(f"\nOverall statistics:")
print(f"  Total StockTwits rows: {df_stats['stocktwits_rows'].sum():,}")
print(f"  Total matched rows: {df_stats['matched_rows'].sum():,}")
print(f"  Overall match rate: {(df_stats['matched_rows'].sum() / df_stats['stocktwits_rows'].sum() * 100):.2f}%")
print(f"\nMerged files saved to: {output_folder}")

# Show example of columns in final output
print(f"\nAbnormal return columns added:")
example_ar_cols = [col for col in df_merged.columns if col.startswith('ar_')]
for col in sorted(example_ar_cols):
    print(f"  - {col}")


Starting year-by-year merge with abnormal return calculations...



Processing years:   7%|▋         | 1/15 [00:11<02:46, 11.87s/it]

  2010: 16,228 tweets -> 10,814 matched (66.5%), 35 AR columns


Processing years:  13%|█▎        | 2/15 [00:24<02:36, 12.07s/it]

  2011: 55,353 tweets -> 31,956 matched (57.7%), 35 AR columns


...

  2022: 17,680,469 tweets -> 11,063,270 matched (62.6%), 35 AR columns


Processing years:  93%|█████████▎| 14/15 [47:39<07:06, 426.33s/it]

  2023: 8,388,025 tweets -> 5,878,223 matched (70.1%), 35 AR columns


Processing years: 100%|██████████| 15/15 [47:56<00:00, 191.77s/it]

  2024: 11,265 tweets -> 5,750 matched (51.0%), 35 AR columns

Merge Summary

 year  stocktwits_rows  crsp_rows  merged_rows  matched_rows  match_rate  ar_columns
 2010            16228    1115087        16252         10814   66.539503          35
 2011            55353    1083669        55431         31956   57.650051          35
 2012           103409    1044142       103533         63431   61.266456          35
 2013           586177    1032391       586578        349452   59.574686          35
 2014          1410083    1059131      1410572        921958   65.360577          35
 2015          2088232    1077765      2088947       1340178   64.155673          35
 2016          3198528    1052799      3199488       1994627   62.342068          35
 2017          6206372    1033631      6208496       4409026   71.016008          35
 2018          7620494    1039644      7623348       5062124   66.402898          35
 2019          7154516    1049914      7157645       5097705   71.220422